# Notebook 07 — Stage 3: carbonate-system integrity checks

**Role in the pipeline:** the first stage with *scientific* QC, not
just data-engineering QC. Reads Stage 2's `enhanced.csv` and runs
row-wise **internal-consistency checks** of the carbonate chemistry:
does `DIC` balance with the sum of its three species (CO₂(aq), HCO₃⁻,
CO₃²⁻)? Does observed `ph_best` agree with calculated `ph_co2sys`
within the expected tolerance? Are the pH scales and units consistent?

```text
06_stage2.ipynb
   └── <stage2_out>/data/enhanced.csv
                             │
                             ▼
                  07_stage3.ipynb           ◄── THIS NOTEBOOK
                     • re-resolve canonical aliases defensively
                     • add helper columns such as sample_month, depth_round_m, lat, lon
                     • run DIC species-sum checks
                     • run pH best-vs-CO2SYS diagnostics
                     • carry solver and input-pair provenance separately
                     • write enhanced.csv, mismatch tables, QC summary, and report
                                                  │
                                                  ▼
                                          08_stage4.ipynb
```

The carbonate-system relation used here is:

```text
DIC = CO₂(aq) + HCO₃⁻ + CO₃²⁻
```

**What Stage 3 does not do:** it does not rebuild `ph_best`,
`ph_co2sys`, `ta_best_umolkg`, or other core chemistry fields. The
integrity checks surface inconsistencies and provenance gaps; the analyst
or Stage 4 audit layer decides how to classify them. Flags are advisory,
not destructive.


## Parameters

Single tagged `parameters` cell. Default `INPUT_CSV` points at the
new short Stage 2 path (`<stage2_out>/data/enhanced.csv`), not the
original's `oa_stage2_outputs\stage2_enhanced.csv` (which the
refactored pipeline never produces).


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Set by run_pipeline.sh during normal pipeline execution.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto oa_pipeline.stage3.STAGE3_DEFAULTS. Use to override
# thresholds, aliases, accepted_ph_scales, or qc_group_keys without editing code.
CONFIG_PATH = None

# --- Stage 3 behaviour ---------------------------------------------
DEPTH_ROUND_DECIMALS = 1
NO_PARQUET = False
DRY_RUN = False


## Setup

Carbonate integrity checks, helper columns, and the QC roll-up all
live in `oa_stage3.py`. The notebook itself is thin orchestration.

Eleven helpers that the original Stage 3 notebook redefined are now
imported from `oa_common.py` and `oa_schema.py`. The `RangePolicy`
dataclass — redefined for the third time in the original — is gone
from here entirely (Stage 3 doesn't actually need it; the audit-flagged
redefinition has no effect on output, just on hygiene).


In [ ]:
from __future__ import annotations

import copy
import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd

try:
    import importlib.metadata as importlib_metadata
    oa_pipeline_version = importlib_metadata.version("oa-pipeline")
except Exception:
    oa_pipeline_version = None

try:
    from IPython.display import display
except Exception:
    display = None

from oa_pipeline.common import (
    deep_update,
    die,
    ensure_dir,
    md_table_from_df,
    normalize_columns,
    utc_stamp,
    write_csv_and_parquet,
    write_json,
    write_text,
)
from oa_pipeline.schema import load_config, assert_ph_scale_consistency  # AUDIT FIX N-6
from oa_pipeline.stage2 import (
    ensure_required_columns,
    ensure_stage2_dirs,
    make_column_inventory,
    make_presence_table,
    materialize_canonical_aliases,
)
from oa_pipeline.stage3 import (
    STAGE3_DEFAULTS,
    CarbonateIntegrityThresholds,
    add_canonical_helper_columns,
    build_qc_summary,
    carbonate_integrity_checks,
)


def as_bool(value) -> bool:
    """Parse Papermill friendly boolean values safely."""
    if isinstance(value, bool):
        return value

    if value is None:
        return False

    text = str(value).strip().lower()

    if text in {"true", "1", "yes", "y"}:
        return True

    if text in {"false", "0", "no", "n", "none", "null", ""}:
        return False

    die(f"Cannot parse boolean parameter value: {value!r}")


## Load input and canonicalise

The same defensive alias resolution as Stage 2: no-op when fed Stage 2's
output, recovers if fed some other CSV that uses different names.


In [ ]:
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)

try:
    DEPTH_ROUND_DECIMALS = int(DEPTH_ROUND_DECIMALS)
except Exception:
    die(f"DEPTH_ROUND_DECIMALS must be an integer, got {DEPTH_ROUND_DECIMALS!r}")

if DEPTH_ROUND_DECIMALS < 0:
    die(f"DEPTH_ROUND_DECIMALS must be >= 0, got {DEPTH_ROUND_DECIMALS}")

if INPUT_CSV is None or str(INPUT_CSV).strip() == "":
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")

if OUT_DIR is None or str(OUT_DIR).strip() == "":
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")

input_csv = Path(INPUT_CSV).expanduser().resolve()

if not input_csv.exists():
    die(
        f"File not found: {input_csv}\n"
        "Did Stage 2 run successfully? Stage 3 reads its enhanced.csv."
    )

if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")

config_path = None
if CONFIG_PATH is not None:
    text = str(CONFIG_PATH).strip()
    if text.lower() not in {"", "none", "null"}:
        config_path = text

user_config, config_source = load_config(config_path)

if config_path:
    config = deep_update(copy.deepcopy(STAGE3_DEFAULTS), user_config)
else:
    config = copy.deepcopy(STAGE3_DEFAULTS)

required_cols = config.get(
    "required_stage3_columns",
    config.get("required_stage2_columns", []),
)

expected_cols = config.get(
    "expected_stage3_columns",
    config.get("expected_stage2_columns", []),
)

out_root = ensure_dir(Path(OUT_DIR).expanduser().resolve())
dirs = ensure_stage2_dirs(out_root)   # same {data,tables,reports,logs} layout

notes: list[str] = []

df = pd.read_csv(input_csv)
df = normalize_columns(df)
df["source_file_stage3"] = str(input_csv)
df["stage3_processed_utc"] = utc_stamp()

df, alias_resolution = materialize_canonical_aliases(
    df,
    config["canonical_aliases"],
    notes,
)

ensure_required_columns(df, required_cols)

presence_df = make_presence_table(
    df,
    required=required_cols,
    expected=expected_cols,
)

missing_opt = presence_df.loc[
    (~presence_df["required"]) & (~presence_df["present"]), "column"
].tolist()

if missing_opt:
    notes.append("Optional expected columns not found: " + ", ".join(missing_opt))

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")
print(f"Config path: {config_path or '(defaults)'}")

if display is not None:
    display(presence_df)


## Add helper columns and column inventory

Helper columns added by `add_canonical_helper_columns`:

- `sample_month` (derived from `sample_date`)
- `depth_round_m` (from `depth_m`, rounded to `DEPTH_ROUND_DECIMALS`)
- `lat`, `lon` (aliases of `latitude_deg`, `longitude_deg`)
- normalised `ph_scale_*_normalized` and `*_unit_normalized` columns
- roll-up flags: `flag_stage2_replicate_conflict_carried`,
  `flag_solver_unknown`, `flag_carbon_input_pair_unknown`


In [ ]:
df = add_canonical_helper_columns(df, notes, depth_round_decimals=DEPTH_ROUND_DECIMALS)
inventory_df = make_column_inventory(df)

if display is not None:
    display(inventory_df.head(20))


## Carbonate-system integrity checks

Three families of tests are defined in `oa_pipeline.stage3.carbonate_integrity_checks`:

1. **DIC species-sum check.** Flags rows where
   `|DIC - (CO2aq + HCO3 + CO3)|` exceeds
   `max(dic_abs_tol, |DIC|*dic_rel_tol)`. The robust variant uses
   MAD outlier detection on the residual distribution instead of
   only a fixed threshold.
2. **pH diagnostic.** Flags rows where `|ph_best - ph_co2sys|`
   exceeds `ph_diag_tol`. The strict variant requires known matching
   pH scales. The robust variant uses MAD on the residual.
3. **Scale + unit consistency.** Per-row flags for missing scale
   context, unexpected pH scales, mismatched pH scales between observed
   and calculated pH, and mismatched units across the DIC species.

A roll-up `flag_any_carbonate_issue` is `True` if any non-strict
chemistry issue fires. A stricter `flag_any_carbonate_issue_strict`
includes severe carbonate consistency problems such as DIC inconsistency,
pH scale mismatch, unexpected pH scale, and scale-aware pH diagnostic
mismatch.

Solver and carbon input-pair provenance are carried separately for
Stage 4 audit decisions. They are not folded into the strict Stage 3
chemistry roll-up.


In [ ]:
thr = CarbonateIntegrityThresholds.from_config(config)
accepted_ph_scales = config.get("accepted_ph_scales", ["total"])

# AUDIT FIX N-6: enforce that the accepted-pH-scale invariant agrees with
# the schema default. Mixing pH scales without a documented conversion
# corrupts carbonate-system calculations (Moras et al. 2023). This turns
# the former comment-only "sync" contract into an enforced check. If a
# cruise/regional config disagrees, this stops the run with a clear error.
try:
    from oa_pipeline.schema import DEFAULT_CONFIG as _SCHEMA_DEFAULTS
    _schema_scales = _SCHEMA_DEFAULTS.get("accepted_ph_scales", ["total"])
except Exception:
    _schema_scales = ["total"]
accepted_ph_scales = assert_ph_scale_consistency(
    accepted_ph_scales,
    _schema_scales,
    source_names=["stage3_config", "schema_default"],
)

flags_df, summary, dic_bad, ph_bad = carbonate_integrity_checks(
    df,
    thr,
    accepted_ph_scales=accepted_ph_scales,
)

# Assign flags into the main frame.
for c in flags_df.columns:
    df[c] = flags_df[c]

print(f"Any carbonate issue : {summary['n_any_carbonate_issue']:,} rows")
print(f"                     (strict: {summary['n_any_carbonate_issue_strict']:,})")
print(f"DIC checkable        : {summary['n_dic_checkable']:,}")
print(f"DIC inconsistent     : {summary['n_dic_inconsistent']:,}")
print(f"pH checkable         : {summary['n_ph_checkable']:,}")
print(f"pH diagnostic mismatch: {summary['n_ph_diag_mismatch']:,}")
print(f"Accepted pH scales   : {accepted_ph_scales}")

if display is not None:
    display(pd.DataFrame([summary]))


## QC summary by group

Aggregates the flag counts to `[cruise_id, transect_id, station_id,
depth_round_m, sample_month]` (by default; configurable via
`qc_group_keys`). Useful for spotting whether the failures cluster
in a particular cruise / station / season.

The summary is also joined back to the row-level frame so each row
carries its group's roll-up counts — handy for downstream filtering.


In [ ]:
qc_df, keys_used = build_qc_summary(df, config.get("qc_group_keys", []))
print(f"QC groups: {len(qc_df):,} | group keys used: {keys_used}")

if keys_used:
    df = df.merge(qc_df, on=keys_used, how="left", suffixes=("", "_grp"))
else:
    for c in qc_df.columns:
        df[c] = qc_df.iloc[0][c]

if display is not None:
    display(qc_df.head(20))


## Quick preview

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "sample_month",
        "station_id", "depth_m", "depth_round_m", "lat", "lon",
        "ta_best_umolkg", "ph_best", "ph_co2sys",
        "dic_best_umol_kg", "co2aq_calc_umol_kg",
        "hco3_calc_umol_kg", "co3_calc_umol_kg",
        "flag_dic_inconsistent", "flag_ph_diag_mismatch",
        "flag_any_carbonate_issue", "flag_any_carbonate_issue_strict",
    ]
    if c in df.columns
]
if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Prepare output paths

Layout — short filenames, identity in the parent folder, no
`<long_input_stem>__stage3_enhanced.csv` prefix.

```
<OUT_DIR>/
    data/
        enhanced.csv                       (and .parquet)   # ◄── Stage 4 input
    tables/
        carbonate_integrity_flags.csv      # id cols + every flag column
        qc_summary_by_group.csv            # per-group flag counts
        dic_species_mismatches.csv         # rows that failed DIC checks
        ph_diag_mismatches.csv             # rows that failed pH checks
        column_inventory.csv
        canonical_presence.csv
        alias_resolution.csv
    reports/
        report.md
    logs/
        manifest.json
        effective_config.json
```


In [ ]:
paths = {
    "enhanced_csv":           dirs["data"]    / "enhanced.csv",
    "enhanced_parquet":       dirs["data"]    / "enhanced.parquet",
    "integrity_flags_csv":    dirs["tables"]  / "carbonate_integrity_flags.csv",
    "qc_summary_csv":         dirs["tables"]  / "qc_summary_by_group.csv",
    "dic_mismatches_csv":     dirs["tables"]  / "dic_species_mismatches.csv",
    "ph_mismatches_csv":      dirs["tables"]  / "ph_diag_mismatches.csv",
    "column_inventory_csv":   dirs["tables"]  / "column_inventory.csv",
    "canonical_presence_csv": dirs["tables"]  / "canonical_presence.csv",
    "alias_resolution_csv":   dirs["tables"]  / "alias_resolution.csv",
    "report_md":              dirs["reports"] / "report.md",
    "manifest_json":          dirs["logs"]    / "manifest.json",
    "effective_config_json":  dirs["logs"]    / "effective_config.json",
}
print(f"Output root: {out_root}")


## Write outputs

In [ ]:
parquet_written = False
parquet_error = None

if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    # Per-table CSVs.
    id_cols = [
        c for c in [
            "record_id", "sample_id", "cruise_id", "transect_id",
            "station_id", "sample_date", "sample_month",
            "depth_round_m", "lat", "lon",
        ]
        if c in df.columns
    ]

    pd.concat(
        [df[id_cols].reset_index(drop=True), flags_df.reset_index(drop=True)],
        axis=1,
    ).to_csv(paths["integrity_flags_csv"], index=False)

    qc_df.to_csv(paths["qc_summary_csv"], index=False)
    inventory_df.to_csv(paths["column_inventory_csv"], index=False)
    presence_df.to_csv(paths["canonical_presence_csv"], index=False)

    # Write mismatch tables deterministically, even when they are empty.
    dic_bad.to_csv(paths["dic_mismatches_csv"], index=False)
    ph_bad.to_csv(paths["ph_mismatches_csv"], index=False)

    alias_df = pd.DataFrame({
        "canonical_column": list(alias_resolution.keys()),
        "resolved_from": list(alias_resolution.values()),
    })
    alias_df.to_csv(paths["alias_resolution_csv"], index=False)

    # Main enhanced frame.
    if NO_PARQUET:
        df.to_csv(paths["enhanced_csv"], index=False)
        parquet_error = "Parquet disabled by user"
    else:
        parquet_written, parquet_error = write_csv_and_parquet(
            df,
            paths["enhanced_csv"],
            paths["enhanced_parquet"],
        )

    write_json(paths["effective_config_json"], config)

    notes_md = "\n".join(f"- {n}" for n in notes) if notes else "- (none)"

    report_md_text = f"""# Stage 3 Preprocessing Report

**Generated:** {utc_stamp()}
**Input:** `{input_csv}`
**Output root:** `{out_root}`
**Rows:** {len(df):,}  **Columns:** {df.shape[1]:,}

## What this stage does
Runs row-wise internal-consistency checks of the carbonate-system
chemistry. **Does not rebuild any core chemistry field.**

Stage 3 separates chemistry consistency flags from provenance audit flags.
Unknown `carbonate_solver` and unknown `carbon_input_pair_used` are carried
forward for Stage 4, but they are not included in the strict Stage 3
chemistry roll-up.

## Carbonate integrity -- DIC species
| Metric | Count |
|--------|------:|
| DIC columns present | {summary["dic_columns_present"]} |
| Rows with all values | {summary["n_dic_values_present"]:,} |
| Checkable rows | {summary["n_dic_checkable"]:,} |
| Unit missing | {summary["n_dic_unit_missing"]:,} |
| Unit mismatch | {summary["n_dic_unit_mismatch"]:,} |
| Non-positive DIC | {summary["n_dic_nonpositive"]:,} |
| Negative species | {summary["n_any_negative_species"]:,} |
| Threshold inconsistent | {summary["n_dic_inconsistent"]:,} |
| Robust outliers | {summary["n_dic_inconsistent_robust"]:,} |

Thresholds: abs={summary["dic_abs_tol"]} umol/kg, rel={summary["dic_rel_tol"]:.3f},
MAD k={summary["dic_mad_k"]}

## Carbonate integrity -- pH diagnostics
| Metric | Count |
|--------|------:|
| pH columns present | {summary["ph_columns_present"]} |
| Rows with values | {summary["n_ph_values_present"]:,} |
| Checkable rows | {summary["n_ph_checkable"]:,} |
| Strict checkable rows | {summary["n_ph_strict_checkable"]:,} |
| Observed pH missing scale | {summary["n_ph_best_missing_scale_context"]:,} |
| Calculated pH missing scale | {summary["n_ph_co2sys_missing_scale_context"]:,} |
| Observed pH unexpected scale | {summary["n_ph_best_scale_unexpected"]:,} |
| Calculated pH unexpected scale | {summary["n_ph_co2sys_scale_unexpected"]:,} |
| Scale mismatch | {summary["n_ph_scale_mismatch"]:,} |
| Threshold mismatch | {summary["n_ph_diag_mismatch"]:,} |
| Strict threshold mismatch | {summary["n_ph_diag_mismatch_strict"]:,} |
| Robust outliers | {summary["n_ph_diag_mismatch_robust"]:,} |

Threshold: tol={summary["ph_diag_tol"]}, MAD k={summary["ph_mad_k"]}

Accepted pH scales: `{accepted_ph_scales}`

## Combined flags
| Flag | Rows |
|------|-----:|
| Stage 2 replicate conflict carried | {summary["n_stage2_replicate_conflict_carried"]:,} |
| Solver unknown | {summary["n_solver_unknown"]:,} |
| Carbon input pair unknown | {summary["n_carbon_input_pair_unknown"]:,} |
| Any carbonate issue | {summary["n_any_carbonate_issue"]:,} |
| Any carbonate issue strict | {summary["n_any_carbonate_issue_strict"]:,} |

## QC summary by group
Group keys used: `{keys_used}`

{md_table_from_df(qc_df.head(20), max_rows=200)}

## Canonical field presence
{md_table_from_df(presence_df, max_rows=200)}

## Column inventory top 30 by missingness
{md_table_from_df(inventory_df.head(30), max_rows=200)}

## Notes
{notes_md}

## Main outputs
- enhanced CSV: `{paths["enhanced_csv"]}`  (Stage 4 reads this)
- integrity flags: `{paths["integrity_flags_csv"]}`
- DIC mismatches: `{paths["dic_mismatches_csv"]}`
- pH mismatches: `{paths["ph_mismatches_csv"]}`
"""

    write_text(paths["report_md"], report_md_text)

    manifest = {
        "notebook": "07_stage3",
        "generated_utc": utc_stamp(),
        "input_csv": str(input_csv),
        "output_root": str(out_root),
        "config_source": config_source,
        "parameters": {
            "INPUT_CSV": str(input_csv),
            "OUT_DIR": str(out_root),
            "CONFIG_PATH": CONFIG_PATH,
            "config_path_resolved": config_path,
            "DEPTH_ROUND_DECIMALS": DEPTH_ROUND_DECIMALS,
            "NO_PARQUET": NO_PARQUET,
            "DRY_RUN": DRY_RUN,
        },
        "thresholds": asdict(thr),
        "accepted_ph_scales": accepted_ph_scales,
        "alias_resolution": alias_resolution,
        "qc_group_keys_used": keys_used,
        "integrity_summary": summary,
        "row_counts": {
            "n_rows": int(len(df)),
            "n_dic_mismatch_rows": int(len(dic_bad)),
            "n_ph_mismatch_rows": int(len(ph_bad)),
            "n_qc_groups": int(len(qc_df)),
            "n_any_carbonate_issue": summary["n_any_carbonate_issue"],
            "n_any_carbonate_issue_strict": summary["n_any_carbonate_issue_strict"],
        },
        "parquet_written": parquet_written,
        "parquet_error": parquet_error,
        "notes": notes,
        "outputs": {k: str(v) for k, v in paths.items()},
        "package_versions": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
            "oa_pipeline": oa_pipeline_version,
        },
    }

    write_json(paths["manifest_json"], manifest)

    print("\nStage 3 complete.")
    print(f"  -> Stage 4 input: {paths['enhanced_csv']}")


## Review written outputs

In [ ]:
if not DRY_RUN:
    outputs_df = pd.DataFrame(
        {"output_name": list(paths.keys()), "path": [str(p) for p in paths.values()]}
    )
    if display is not None:
        display(outputs_df)
        if not dic_bad.empty:
            print("\nDIC mismatches:")
            display(dic_bad.head(20))
        if not ph_bad.empty:
            print("\npH mismatches:")
            display(ph_bad.head(20))
    else:
        print(outputs_df.to_string(index=False))
